In [ ]:
import streamlit as st
import pandas as pd
from datetime import datetime
import os

st.set_page_config(page_title="Подтверждение аптек", layout="wide")
st.title("Система набора аптек для таргет базы")

# ====================== Файлы ======================
ADMINS_FILE = "admin.xlsx"
AUTH_FILE = "Log.xlsx"
PLACES_FILE = "Trade.xlsx"
SUBMISSIONS_FILE = "submissions.xlsx"

# ====================== Разделение на линии ======================
PRODUCTS_G = ["Сафорель", "Стимол", "Стрезам"]
PRODUCTS_BASE = ["Энтерол", "Альфлорекс", "Отипакс", "А-церумен", "Сафорель"]
PRODUCTS_RD = ["Энтерол", "Альфлорекс", "Отипакс", "А-церумен", "Стрезам", "Сафорель", "Стимол"]
ALL_PRODUCTS = PRODUCTS_RD #нужно для проверки загружаемого файла на ниличие всех необходимых колонок

# ====================== Указания при загрузке ======================
def load_admins():
    if not os.path.exists(ADMINS_FILE):
        return pd.DataFrame(columns=["login", "pass-code"])
    return pd.read_excel(ADMINS_FILE)

def load_auth():
    if not os.path.exists(AUTH_FILE):
        return pd.DataFrame()
    return pd.read_excel(AUTH_FILE)

def load_places():
    if not os.path.exists(PLACES_FILE):
        return pd.DataFrame()
    return pd.read_excel(PLACES_FILE)

def load_submissions():
    if os.path.exists(SUBMISSIONS_FILE):
        return pd.read_excel(SUBMISSIONS_FILE)
    return pd.DataFrame(columns=[
        "submission_time", "login", "id", "Город", "Организация", "Адрес",
        "Энтерол", "Альфлорекс", "Отипакс", "Таргет", "А-церумен",
        "Стрезам", "Сафорель", "Стимол", "ИНН", "notes"
    ])

def save_uploaded_excel(uploaded_file, target_path: str) -> bool:
    try:
        df = pd.read_excel(uploaded_file)
        df.to_excel(target_path, index=False)
        return True
    except Exception as e:
        st.error(f"Не удалось сохранить файл: {e}")
        return False

def get_user_cities(user: dict) -> list:
    cities = []
    for col in ["Город", "Город 2", "Город 3", "Город 4", "Город 5", "Город 6", "Город 7", "Город 8", "Город 9", "Город 10", "Город 11", "Город 12", "Город 13", "Город 14", "Город 15"]:
        val = user.get(col, None)
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        val = str(val).strip()
        if val and val.lower() not in ("nan", "none", ""):
            cities.append(val.lower())
    if not cities:
        lower_map = {str(k).strip().lower(): v for k, v in user.items()}
        for col in ["город", "город 2", "город 3", "город 4", "город 5", "город 6", "город 7", "Город 8", "Город 9", "Город 10", "Город 11", "Город 12", "Город 13", "Город 14", "Город 15"]:
            val = lower_map.get(col)
            if val is None or (isinstance(val, float) and pd.isna(val)):
                continue
            val = str(val).strip()
            if val and val.lower() not in ("nan", "none", ""):
                cities.append(val.lower())
    seen, result = set(), []
    for c in cities:
        if c not in seen:
            seen.add(c)
            result.append(c)
    return result

def get_user_regions(user: dict) -> list:
    """For rd users – collect Регион / Регион 2 …"""
    regions = []
    for col in ["Регион", "Регион 2", "Регион 3", "Регион 4", "Регион 5", "Регион 6", "Регион 7"]:
        val = user.get(col, None)
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        val = str(val).strip()
        if val and val.lower() not in ("nan", "none", ""):
            regions.append(val.lower())
    if not regions:
        lower_map = {str(k).strip().lower(): v for k, v in user.items()}
        for col in ["регион", "регион 2", "регион 3", "регион 4", "регион 5", "регион 6", "регион 7",
                    "region", "region 2", "region 3"]:
            val = lower_map.get(col)
            if val is None or (isinstance(val, float) and pd.isna(val)):
                continue
            val = str(val).strip()
            if val and val.lower() not in ("nan", "none", ""):
                regions.append(val.lower())
    seen, result = set(), []
    for r in regions:
        if r not in seen:
            seen.add(r)
            result.append(r)
    return result

def get_user_category(user: dict) -> str:
    """Return 'g', 'base' or 'rd'. Default = 'base'."""
    for key in ["Категория", "категория", "category", "Category", "role", "Role"]:
        val = user.get(key)
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        val = str(val).strip().lower()
        if val in ("g", "г"):
            return "g"
        if val in ("base", "базовый", "база"):
            return "base"
        if val in ("rd", "рд"):
            return "rd"
    # fallback – try lower-case keys
    lower_map = {str(k).strip().lower(): v for k, v in user.items()}
    for key in ["категория", "category", "role"]:
        val = lower_map.get(key)
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        val = str(val).strip().lower()
        if val in ("g", "г"):
            return "g"
        if val in ("base", "базовый", "база"):
            return "base"
        if val in ("rd", "рд"):
            return "rd"
    return "base"

def get_visible_products(category: str) -> list:
    if category == "g":
        return PRODUCTS_G
    if category == "rd":
        return PRODUCTS_RD
    return PRODUCTS_BASE

def get_place_id_series(df: pd.DataFrame) -> pd.Series:
    if "TradepointId" in df.columns:
        return df["TradepointId"].astype(str).str.strip()
    if "id" in df.columns:
        return df["id"].astype(str).str.strip()
    return pd.Series([""] * len(df), index=df.index)

def normalize_target(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    s = s.replace({
        "1": "да", "0": "нет",
        "true": "да", "false": "нет",
        "yes": "да", "no": "нет",
        "y": "да", "n": "нет",
        "да": "да", "нет": "нет",
    })
    return s.apply(lambda x: x if x in ("да", "нет") else "нет")

def get_region_series(df: pd.DataFrame) -> pd.Series:
    if "Регион" in df.columns:
        return df["Регион"].astype(str).str.strip().str.lower()
    if "region" in df.columns:
        return df["region"].astype(str).str.strip().str.lower()
    return pd.Series([""] * len(df), index=df.index)

# ====================== SESSION STATE ======================
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "current_user" not in st.session_state:
    st.session_state.current_user = None
if "is_admin" not in st.session_state:
    st.session_state.is_admin = False

# ====================== Вход в систему ======================
def login_page():
    st.subheader("Вход")
    role = st.radio("Войти как", ["Пользователь", "Админ"], horizontal=True)
    col1, col2 = st.columns(2)
    with col1:
        login_input = st.text_input("Login").strip()
    with col2:
        pass_input = st.text_input("Pass-code", type="password").strip()

    if st.button("Login", type="primary"):
        if not login_input or not pass_input:
            st.error("Введите login и pass-code.")
            return

        if role == "Админ":
            admins = load_admins()
            if admins.empty:
                st.error(f"Файл админов '{ADMINS_FILE}' не найден или пуст.")
                return
            df = admins.copy()
            df["login"] = df["login"].astype(str).str.strip()
            df["pass-code"] = df["pass-code"].astype(str).str.strip()
            row = df[
                (df["login"].str.lower() == login_input.lower()) &
                (df["pass-code"] == pass_input)
            ]
            if row.empty:
                st.error("Неверный admin login или pass-code")
                return
            st.session_state.logged_in = True
            st.session_state.is_admin = True
            st.session_state.current_user = row.iloc[0].to_dict()
            st.success(f"Добро пожаловать, Admin **{row.iloc[0]['login']}**!")
            st.rerun()
        else:
            auth = load_auth()
            if auth.empty:
                st.error("Данные пользователей ещё не загружены. Попросите администратора загрузить Log.xlsx.")
                return

            cols = {c.lower().strip(): c for c in auth.columns}
            login_col = cols.get("login")
            pass_col = cols.get("pass-code") or cols.get("passcode") or cols.get("password")
            if not login_col or not pass_col:
                st.error("В файле пользователей нужны колонки: login, pass-code")
                return

            df = auth.copy()
            df["_login"] = df[login_col].astype(str).str.strip()
            df["_pass"] = df[pass_col].astype(str).str.strip()
            row = df[
                (df["_login"].str.lower() == login_input.lower()) &
                (df["_pass"] == pass_input)
            ]
            if row.empty:
                st.error("Неверный login или pass-code")
                return

            user_dict = row.iloc[0].to_dict()
            normalized = {}
            for k, v in user_dict.items():
                normalized[str(k).strip()] = v
                normalized[str(k).strip().lower()] = v

            st.session_state.logged_in = True
            st.session_state.is_admin = False
            st.session_state.current_user = normalized
            st.success(f"Добро пожаловать, **{login_input}**!")
            st.rerun()

if not st.session_state.logged_in:
    login_page()
    st.stop()

user = st.session_state.current_user
is_admin = st.session_state.is_admin
current_login = str(user.get("login", "")).strip()
user_category = get_user_category(user) if not is_admin else "admin"
visible_products = get_visible_products(user_category) if not is_admin else ALL_PRODUCTS

# ====================== Поисковая колонка ======================
st.sidebar.success(current_login.title() if current_login else "User")
if is_admin:
    st.sidebar.success("ADMIN ACCESS")
else:
    st.sidebar.info(f"Категория: **{user_category.upper()}**")
    if user_category == "rd":
        user_regions_sb = get_user_regions(user)
        if user_regions_sb:
            st.sidebar.info("Регионы:\n" + "\n".join(r.upper() for r in user_regions_sb))
        else:
            st.sidebar.warning("Регионы не назначены")
    else:
        user_cities_sb = get_user_cities(user)
        if user_cities_sb:
            st.sidebar.info("Города:\n" + "\n".join(c.upper() for c in user_cities_sb))
        else:
            st.sidebar.warning("Города не назначены")

if st.sidebar.button("Выйти из аккаунта"):
    st.session_state.clear()
    st.rerun()

# ====================== Доступ администратора ==================================
if is_admin:
    st.subheader("Панель администратора")
    tab_upload, tab_subs, tab_preview = st.tabs(
        ["Загрузка данных", "Подтверждения", "Просмотр данных"]
    )

    with tab_upload:
        st.markdown("### Загрузка файлов для пользователей")
        c1, c2 = st.columns(2)
        with c1:
            st.markdown(f"**1. Пользователи** → `{AUTH_FILE}`")
            st.caption(
                "Обязательно: `login`, `pass-code`, `Город`, `Категория` (G / base / rd)\n\n"
                "Для обычных пользователей: `Город 2`…`Город 7`\n\n"
                "Для rd: `Регион`, `Регион 2`…"
            )
            users_file = st.file_uploader("Загрузить Log (пользователи)", type=["xlsx", "xls"], key="upload_users")
            if users_file is not None:
                if st.button("Сохранить пользователей", type="primary", key="save_users"):
                    if save_uploaded_excel(users_file, AUTH_FILE):
                        st.success(f"Файл сохранён: {AUTH_FILE}")
                        st.cache_data.clear()

        with c2:
            st.markdown(f"**2. Организации** → `{PLACES_FILE}`")
            st.caption(
                "Обязательно: `TradepointId` (или `id`), `Город`, `Организация`, `Адрес`\n\n"
                "Для rd: колонка `Регион`\n\n"
                "Опционально: продажи, `Таргет`, `КАС`, `ИНН`"
            )
            places_file = st.file_uploader("Загрузить Target (места)", type=["xlsx", "xls"], key="upload_places")
            if places_file is not None:
                if st.button("Сохранить места", type="primary", key="save_places"):
                    if save_uploaded_excel(places_file, PLACES_FILE):
                        st.success(f"Файл сохранён: {PLACES_FILE}")
                        st.cache_data.clear()

        st.divider()
        s1, s2, s3 = st.columns(3)
        s1.metric("Пользователи", "Готов" if os.path.exists(AUTH_FILE) else "Нет файла")
        s2.metric("Места", "Готов" if os.path.exists(PLACES_FILE) else "Нет файла")
        s3.metric("Подтверждений", len(load_submissions()))

        # Количество строк в загруженных таблицах
        if os.path.exists(AUTH_FILE):
            try:
                st.caption(f"Пользователей: **{len(pd.read_excel(AUTH_FILE))}**")
            except Exception:
                pass
        if os.path.exists(PLACES_FILE):
            try:
                st.caption(f"Организаций: **{len(pd.read_excel(PLACES_FILE))}**")
            except Exception:
                pass
        if os.path.exists(SUBMISSIONS_FILE):
            try:
                st.caption(f"Подтверждений (строк): **{len(pd.read_excel(SUBMISSIONS_FILE))}**")
            except Exception:
                pass

    with tab_subs:
        st.markdown("### Все подтверждения")
        submissions = load_submissions()
        if os.path.exists(SUBMISSIONS_FILE):
            try:
                st.caption(f"Подтверждений (строк): **{len(pd.read_excel(SUBMISSIONS_FILE))}**")
            except Exception:
                pass
        if submissions.empty:
            st.info("Нет подтверждённых организаций.")
        else:
            st.dataframe(submissions.sort_values("submission_time", ascending=False),
                         use_container_width=True, hide_index=True)
            st.download_button(
                "Скачать подтверждения (CSV)",
                submissions.to_csv(index=False).encode(),
                f"submissions_{datetime.now().strftime('%Y%m%d')}.csv",
                "text/csv"
            )
            st.divider()
            c1, c2 = st.columns(2)
            c1.metric("Всего подтверждений", len(submissions))
            by_user = (
                submissions.groupby(submissions["login"].astype(str))
                .size().reset_index(name="выбрано")
                .sort_values("выбрано", ascending=False)
            )
            c2.dataframe(by_user, use_container_width=True, hide_index=True)

    with tab_preview:
        st.markdown(f"### {AUTH_FILE}")
        auth = load_auth()
        if auth.empty:
            st.warning("Файл пользователей ещё не загружен.")
        else:
            st.caption(f"Пользователей: **{len(auth)}**")
            st.dataframe(auth, use_container_width=True, hide_index=True)
        st.markdown(f"### {PLACES_FILE}")
        places = load_places()
        if places.empty:
            st.warning("Файл мест ещё не загружен.")
        else:
            st.caption(f"Организаций: **{len(places)}**")
            st.dataframe(places, use_container_width=True, hide_index=True)

    st.stop()

# ====================== Пользовательский доступ ====================================
places_df = load_places()
if places_df.empty:
    st.error("Данные организаций ещё не загружены. Обратитесь к администратору.")
    st.stop()

if "TradepointId" not in places_df.columns and "id" not in places_df.columns:
    st.error("В Trade.xlsx нет колонки TradepointId (или id).")
    st.stop()

# ---------- Региональный фильтр ----------
if user_category == "rd":
    user_scopes = get_user_regions(user)
    scope_label = "регионы"
    if not user_scopes:
        st.error("Вам не назначены Регионы. Обратитесь к администратору.")
        st.stop()
    if "Регион" not in places_df.columns and "region" not in places_df.columns:
        st.error("В Trade.xlsx нет колонки «Регион». Обратитесь к администратору.")
        st.stop()
    region_series = get_region_series(places_df)
    filtered = places_df[region_series.isin(user_scopes)].copy()
else:
    user_scopes = get_user_cities(user)
    scope_label = "города"
    if not user_scopes:
        st.error("Вам не назначены Города. Обратитесь к администратору.")
        st.stop()
    if "Город" not in places_df.columns:
        st.error("В Trade.xlsx нет колонки «Город».")
        st.stop()
    filtered = places_df[
        places_df["Город"].astype(str).str.strip().str.lower().isin(user_scopes)
    ].copy()

if filtered.empty:
    st.warning(f"В ваших {scope_label} не найдено организаций.")
    st.stop()

# ---------- Фильтры ----------
st.sidebar.subheader("Поиск и фильтры")

sort_options = ["По умолчанию"]
for p in visible_products:
    sort_options.append(f"{p} (High to Low)")
sort_options.append("Total Sales (High to Low)")

sort_option = st.sidebar.selectbox("Сортировка", sort_options)
show_only_high = st.sidebar.checkbox("Только с высокими продажами", value=False)
show_only_available = st.sidebar.checkbox("Только доступные организации", value=True)
target_filter = st.sidebar.selectbox("Таргет", ["Все", "Да", "Нет"])
search_address = st.sidebar.text_input("Поиск по адресу", placeholder="Адрес...").strip().lower()
search_kas = st.sidebar.text_input("Поиск по КАС", placeholder="КАС...").strip().lower()

st.subheader(f"Выбор организаций — {', '.join(s.upper() for s in user_scopes)} | Категория: {user_category.upper()}")
if user_category == "rd":
    st.info("Категория RD: вы видите таблицу организаций, но **не можете выбирать** новые. Можно искать по адресу и отменять существующие выборы.")

submissions = load_submissions()

filtered["id"] = get_place_id_series(filtered)
filtered = filtered[
    filtered["id"].notna() & (filtered["id"] != "") & (filtered["id"].str.lower() != "nan")
]
if filtered.empty:
    st.error("Нет корректных id.")
    st.stop()

if "Таргет" not in filtered.columns:
    filtered["Таргет"] = "нет"
filtered["Таргет"] = normalize_target(filtered["Таргет"])

for col in ALL_PRODUCTS:
    if col not in filtered.columns:
        filtered[col] = 0

# Total_Sales only on visible products for this category
filtered["Total_Sales"] = sum(
    pd.to_numeric(filtered[c], errors="coerce").fillna(0) for c in visible_products
)

if "КАС" not in filtered.columns:
    filtered["КАС"] = ""
if "ИНН" not in filtered.columns:
    filtered["ИНН"] = ""

# already chosen map
if not submissions.empty and "id" in submissions.columns:
    taken_map = (
        submissions.assign(key=submissions["id"].astype(str).str.strip())
        .groupby("key")["login"]
        .apply(lambda x: ", ".join(sorted(set(x.astype(str)))))
        .to_dict()
    )
    chosen_by_user = int((submissions["login"].astype(str).str.strip() == current_login).sum())
    chosen_total = len(submissions)
else:
    taken_map = {}
    chosen_by_user = 0
    chosen_total = 0

filtered["key"] = filtered["id"]
filtered["Already_Chosen"] = filtered["key"].isin(taken_map.keys())
filtered["Status"] = filtered["key"].map(
    lambda k: f"Уже выбрано: {taken_map[k]}" if k in taken_map else "Доступно"
)

if target_filter == "Да":
    filtered = filtered[filtered["Таргет"] == "да"]
elif target_filter == "Нет":
    filtered = filtered[filtered["Таргет"] == "нет"]

if search_address and "Адрес" in filtered.columns:
    filtered = filtered[filtered["Адрес"].astype(str).str.lower().str.contains(search_address, na=False)]

if search_kas:
    filtered = filtered[filtered["КАС"].astype(str).str.lower().str.contains(search_kas, na=False)]

# sorting
if sort_option.endswith("(High to Low)") and sort_option != "Total Sales (High to Low)":
    prod = sort_option.replace(" (High to Low)", "")
    if prod in filtered.columns:
        filtered = filtered.sort_values(prod, ascending=False)
elif sort_option == "Total Sales (High to Low)":
    filtered = filtered.sort_values("Total_Sales", ascending=False)
else:
    # default order by scope
    if user_category == "rd":
        order_map = {r: i for i, r in enumerate(user_scopes)}
        filtered["_rank"] = get_region_series(filtered).map(order_map)
    else:
        order_map = {c: i for i, c in enumerate(user_scopes)}
        filtered["_rank"] = filtered["Город"].astype(str).str.strip().str.lower().map(order_map)
    filtered = filtered.sort_values("_rank").drop(columns=["_rank"], errors="ignore")

if show_only_high and len(filtered) > 0:
    filtered = filtered[filtered["Total_Sales"] > filtered["Total_Sales"].mean()]

if show_only_available:
    filtered = filtered[~filtered["Already_Chosen"]]

# ---------- Категории ----------
m1, m2, m3, m4 = st.columns(4)
if user_category == "rd":
    m1.metric(f"Организаций в ваших {scope_label}", len(places_df[get_region_series(places_df).isin(user_scopes)]))
else:
    m1.metric(f"Организаций в ваших {scope_label}",
              len(places_df[places_df["Город"].astype(str).str.strip().str.lower().isin(user_scopes)]))
m2.metric("Уже выбрано", sum(1 for k in taken_map if k in set(filtered["id"])))
m3.metric("Доступно сейчас", len(filtered))
m4.metric("Выбрано вами", chosen_by_user)
st.caption(f"Всего подтверждений в системе: **{chosen_total}** | Ваша категория: **{user_category.upper()}**")

# ---------- TABLE (Trade) ----------
filtered = filtered.reset_index(drop=True)

visible_cols = ["Status"]
if user_category == "rd":
    if "Регион" in filtered.columns:
        visible_cols.append("Регион")
    elif "region" in filtered.columns:
        visible_cols.append("region")
else:
    visible_cols.append("Город")

visible_cols += ["Организация", "Адрес", "КАС"]
visible_cols += [p for p in visible_products if p in filtered.columns]
visible_cols += ["Total_Sales", "Таргет"]
visible_cols = [c for c in visible_cols if c in filtered.columns]

if user_category == "rd":
    # RD: только просмотр, без выбора
    st.dataframe(
        filtered[visible_cols],
        use_container_width=True,
        hide_index=True
    )
else:
    # Обычные пользователи: с выбором
    filtered["Select"] = False
    visible_cols = ["Select"] + visible_cols

    edited_df = st.data_editor(
        filtered[visible_cols],
        hide_index=True,
        use_container_width=True,
        disabled=[c for c in visible_cols if c != "Select"],
        column_config={
            "Select": st.column_config.CheckboxColumn("Выбрать", default=False),
            "Status": st.column_config.TextColumn("Статус", width="medium"),
            "Адрес": st.column_config.TextColumn("Адрес", width="large"),
            "КАС": st.column_config.TextColumn("КАС", width="medium"),
        },
        key="main_selection_editor"
    )

    notes = st.text_area("Дополнительные заметки (необязательно)", height=80)

    if st.button("Подтвердить выбранные организации", type="primary", use_container_width=True):
        selected_mask = edited_df["Select"] == True
        if not selected_mask.any():
            st.warning("Выберите хотя бы одну организацию.")
        else:
            selected_full = filtered.loc[edited_df.index[selected_mask]].copy()
            latest = load_submissions()
            taken_keys = set(latest["id"].astype(str).str.strip()) if not latest.empty and "id" in latest.columns else set()

            selected_full["key"] = selected_full["id"].astype(str).str.strip()
            conflict = selected_full[selected_full["key"].isin(taken_keys)]
            valid = selected_full[~selected_full["key"].isin(taken_keys)]

            if not conflict.empty:
                st.error(f"{len(conflict)} организация(и) уже заняты и не были сохранены.")

            if valid.empty:
                st.warning("Нет доступных организаций для сохранения.")
            else:
                new_rows = []
                for _, row in valid.iterrows():
                    new_rows.append({
                        "submission_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        "login": current_login,
                        "id": str(row["id"]).strip(),
                        "Город": row.get("Город"),
                        "Организация": row.get("Организация"),
                        "Адрес": row.get("Адрес"),
                        "Энтерол": row.get("Энтерол"),
                        "Альфлорекс": row.get("Альфлорекс"),
                        "Отипакс": row.get("Отипакс"),
                        "Таргет": row.get("Таргет", "нет"),
                        "А-церумен": row.get("А-церумен"),
                        "Стрезам": row.get("Стрезам"),
                        "Сафорель": row.get("Сафорель"),
                        "Стимол": row.get("Стимол"),
                        "ИНН": row.get("ИНН", ""),
                        "notes": notes,
                    })
                new_df = pd.DataFrame(new_rows)
                if os.path.exists(SUBMISSIONS_FILE):
                    existing = pd.read_excel(SUBMISSIONS_FILE)
                    final = pd.concat([existing, new_df], ignore_index=True)
                else:
                    final = new_df
                final.to_excel(SUBMISSIONS_FILE, index=False)
                st.success(f"Успешно подтверждено **{len(new_rows)}** организация(й)!")
                st.rerun()

# ====================== Отмена выбора ================================

st.divider()
st.subheader("Отмена выбора")

if user_category == "rd":
    st.caption("Категория RD — вы можете отменять выбор **любых** пользователей.")
    view_subs = submissions.copy() if not submissions.empty else pd.DataFrame()
else:
    st.caption("Вы можете отменять только **свои** выборы.")
    view_subs = submissions[submissions["login"].astype(str).str.strip() == current_login].copy() if not submissions.empty else pd.DataFrame()

# Дополнительный поиск по адресу в уже выбранных
search_address_undo = st.text_input("Поиск по адресу в уже выбранных", placeholder="Адрес...", key="undo_search_address").strip().lower()

if view_subs.empty:
    st.info("Нет записей для отмены.")
else:
    if search_address_undo and "Адрес" in view_subs.columns:
        view_subs = view_subs[view_subs["Адрес"].astype(str).str.lower().str.contains(search_address_undo, na=False)]

    if view_subs.empty:
        st.info("Нет записей, соответствующих поиску.")
    else:
        view_subs = view_subs.sort_values("submission_time", ascending=False).reset_index(drop=True)
        view_subs["Отменить"] = False

        undo_cols = [c for c in ["Отменить", "submission_time", "login", "id", "Город", "Организация", "Адрес", "Таргет", "notes"] if c in view_subs.columns]

        edited_undo = st.data_editor(
            view_subs[undo_cols],
            hide_index=True,
            use_container_width=True,
            disabled=[c for c in undo_cols if c != "Отменить"],
            column_config={
                "Отменить": st.column_config.CheckboxColumn("Отменить", default=False),
                "submission_time": st.column_config.TextColumn("Время", width="medium"),
                "Адрес": st.column_config.TextColumn("Адрес", width="large"),
            },
            key="undo_editor"
        )

        if st.button("Отменить отмеченные организации", type="secondary", use_container_width=True):
            to_cancel = edited_undo["Отменить"] == True
            if not to_cancel.any():
                st.warning("Отметьте хотя бы одну запись.")
            else:
                cancel_df = view_subs.loc[edited_undo.index[to_cancel]]
                ids_to_cancel = set(cancel_df["id"].astype(str).str.strip())

                latest = load_submissions()
                if latest.empty:
                    st.warning("Файл пуст.")
                else:
                    if user_category == "rd":
                        # РД может отменять любой выбор
                        mask_keep = ~latest["id"].astype(str).str.strip().isin(ids_to_cancel)
                    else:
                        # обычный пользователь отменяет только свое
                        mask_keep = ~(
                            (latest["login"].astype(str).str.strip() == current_login) &
                            (latest["id"].astype(str).str.strip().isin(ids_to_cancel))
                        )
                    remaining = latest[mask_keep].copy()
                    remaining.to_excel(SUBMISSIONS_FILE, index=False)
                    st.success(f"Отменено записей: **{len(cancel_df)}**. Организации снова доступны.")
                    st.rerun()

Writing app.py
